# 06b — Rockfall Runout by Rock Size: A Monte-Carlo Experiment

**Short course:** *Geomorphological Hazards of Slopes* &nbsp;•&nbsp; University of Silesia in Sosnowiec &nbsp;•&nbsp; 2 ECTS

*Lecturer: Ola Fredin*

---

Field observations consistently show that **large blocks travel farther than small ones** on the same slope. A truck-sized boulder ploughs through scree, flattens small trees, and rolls hundreds of metres beyond the talus toe; a fist-sized cobble barely clears the cliff face. This notebook isolates the mass effect using the same lumped-mass simulator as Notebook 06, extending the Monte-Carlo framework to draw rocks from five size classes and examine how block mass shapes the runout distribution.

## About this notebook

**Learning objectives.** By the end of this notebook the student will be able to:

1. Explain *why* block mass affects runout even though mass cancels in the equations of free flight.
2. Describe how normal and tangential restitution coefficients vary with block size, and why.
3. Interpret Monte-Carlo runout distributions stratified by size class, including box-plot and exceedance-curve representations.
4. Discuss the implications for hazard zoning when the source cliff can produce blocks of mixed size.

**Relationship to Notebook 06.** The terrain, the simulator, and the probabilistic framework are unchanged. The only extension is that $R_n$ and $R_t$ now depend on the randomly sampled block mass. Notebook 06 need not be open — this notebook is self-contained.

## 1. Does mass matter in a lumped-mass model?

At first glance it should not. Between bounces the block is in free flight under gravity, and Newton's second law gives

$$
\mathbf{a} = \frac{\mathbf{F}}{m} = \frac{m\mathbf{g}}{m} = \mathbf{g}.
$$

Mass cancels. A 50 kg cobble and a 5 000 kg boulder launched with the same initial velocity follow identical parabolas between bounces — just like Galileo's cannonballs.

The difference enters at **impact**. When a block hits the slope, energy is dissipated through:

- **Inelastic deformation of the surface** — gravel compaction, soil shear, spalling of bedrock. A heavier block punches deeper, dissipating more energy *normally*, so its effective $R_n$ is actually *lower* for soft substrates.
- **Surface irregularity and small obstacles** — pebbles, roots, micro-topography. A small block is redirected or trapped by features it cannot override. A large block steamrolls them. This raises the effective $R_t$ for large blocks on rough terrain.
- **Vegetation drag** — saplings and brush absorb energy from a small block disproportionately; a large block snaps them. Again, $R_t$ is effectively higher for large blocks in vegetated terrain.

The net empirical result, confirmed by back-analysis of historical events (Dorren 2003; Bourrier et al. 2009), is that **larger blocks tend to have higher $R_t$ and, on soil/talus terrain, comparable or slightly lower $R_n$** — but the $R_t$ increase dominates, so they travel farther.

### Size classes used in this notebook

We define five classes that span the practical range from field-mappable boulders down to the cobbles that make up talus scree:

| Class | Mass range (kg) | Representative $m$ (kg) | Equivalent sphere diameter |
|---|---|---|---|
| XS | 10 – 100 | 50 | ~0.25 m |
| S | 100 – 500 | 250 | ~0.50 m |
| M | 500 – 1 500 | 1 000 | ~0.80 m |
| L | 1 500 – 3 000 | 2 000 | ~1.05 m |
| XL | 3 000 – 10 000 | 5 500 | ~1.50 m |

Density assumed 2 650 kg m⁻³ (granite/gneiss). The restitution parameterisation follows Bourrier et al. (2009) for an alpine talus surface.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from style import apply_style, COLORS, save_figure
apply_style()

G = 9.81          # m/s²
RNG = np.random.default_rng(seed=2024)   # reproducible, independent RNG

print("Libraries loaded.")

## 2. Terrain and simulator

The same synthetic Norwegian fjord-slope profile from Notebook 06 is used throughout. The source is at the cliff top (x = 0, y = 600 m). The profile transitions from a near-vertical headscarp, through a steep talus, to a gentle outwash apron that reaches sea level at x ≈ 1 300 m.

In [ ]:
from scipy.interpolate import PchipInterpolator

# Terrain profile (synthetic Norwegian fjord-slope)
_key_x = np.array([   0.0,   30.0,  120.0,  280.0,  500.0,  700.0,  900.0,  1300.0])
_key_y = np.array([ 600.0,  500.0,  300.0,  150.0,   80.0,   40.0,   15.0,     0.0])
_interp  = PchipInterpolator(_key_x, _key_y)
terrain_x = np.linspace(_key_x[0], _key_x[-1], 400)
terrain_y = _interp(terrain_x)

source_x, source_y = 0.0, 600.0

# Quick plot to confirm terrain looks right
fig, ax = plt.subplots(figsize=(9.5, 3.8))
ax.fill_between(terrain_x, terrain_y, -30, color=COLORS["soil"], alpha=0.25)
ax.plot(terrain_x, terrain_y, color=COLORS["soil"], lw=2.5)
ax.plot([source_x], [source_y], "o", ms=12, color=COLORS["fail"], label="source")
ax.set_xlim(0, 1300); ax.set_ylim(-30, 680)
ax.set_xlabel("horizontal distance [m]")
ax.set_ylabel("elevation [m]")
ax.set_title("Terrain profile — synthetic Norwegian fjord-slope")
ax.legend()
save_figure(fig, "06b_terrain")
plt.show()

In [ ]:
def simulate_rockfall(x0, y0, vx0, vy0, terrain_x, terrain_y,
                      R_n=0.30, R_t=0.80, dt=0.01, v_min=0.7, max_steps=50_000):
    """2-D lumped-mass rockfall simulator (unchanged from Notebook 06).

    Parameters
    ----------
    x0, y0   : initial position [m]
    vx0, vy0 : initial velocity [m/s]
    R_n      : normal restitution coefficient  (0 < R_n < 1)
    R_t      : tangential restitution coefficient (0 < R_t < 1)

    Returns
    -------
    xs, ys   : trajectory arrays
    bounces  : (n, 2) array of impact (x, y) locations
    """
    def y_terrain(x):
        return np.interp(x, terrain_x, terrain_y)

    def slope_dy_dx(x):
        delta = 0.5
        return (y_terrain(x + delta) - y_terrain(x - delta)) / (2.0 * delta)

    xs, ys = [x0], [y0]
    bounces = []
    x, y, vx, vy = x0, y0, vx0, vy0

    for _ in range(max_steps):
        x_new = x  + vx * dt
        y_new = y  + vy * dt
        vy_new = vy - G * dt

        if y_new <= y_terrain(x_new):
            y_impact = y_terrain(x_new)
            slope = slope_dy_dx(x_new)
            n_mag = np.sqrt(1.0 + slope * slope)
            nx, ny = -slope / n_mag, 1.0 / n_mag

            v_n  = vx * nx + vy * ny
            v_tx = vx - v_n * nx
            v_ty = vy - v_n * ny

            v_n_new = -R_n * v_n
            vx = R_t * v_tx + v_n_new * nx
            vy = R_t * v_ty + v_n_new * ny

            bounces.append((x_new, y_impact))
            speed = np.hypot(vx, vy)
            local_slope_deg = abs(np.degrees(np.arctan(slope)))
            v_n_after = vx * nx + vy * ny
            bounce_height_apex = max(v_n_after, 0.0) ** 2 / (2.0 * G)

            if ((speed < v_min and local_slope_deg < 25.0)
                    or (bounce_height_apex < 0.05 and speed < 3.0)
                    or x_new > terrain_x[-1] - 5.0):
                xs.append(x_new); ys.append(y_impact)
                break

            x, y = x_new, y_impact + 0.05
        else:
            x, y, vy = x_new, y_new, vy_new
        xs.append(x); ys.append(y)

    return np.array(xs), np.array(ys), np.array(bounces) if bounces else np.empty((0, 2))


print("Simulator ready.")

## 3. Mass-dependent restitution coefficients

Each size class is assigned a **mean and standard deviation** for both $R_n$ and $R_t$. Values are based on back-analysis of instrumented rockfall experiments on alpine talus (Bourrier et al. 2009; Dorren et al. 2006):

- $R_n$ increases slightly with mass (heavier blocks are stiffer in collision) but the effect is modest.
- $R_t$ increases more strongly with mass (heavier blocks override surface roughness and vegetation).
- Both parameters are drawn from truncated normal distributions to avoid physically impossible values.

The resulting spread in each class reflects genuine field variability in block shape, impact geometry, and local surface conditions.

In [ ]:
# ---------------------------------------------------------------------------
# Size-class definitions
# ---------------------------------------------------------------------------
SIZE_CLASSES = [
    {
        "label":    "XS  (10–100 kg)",
        "short":    "XS",
        "mass_lo":  10,
        "mass_hi":  100,
        "m_rep":    50,       # representative mass for table / printing
        "Rn_mu":   0.27,     # mean normal restitution
        "Rn_sig":  0.05,
        "Rn_lo":   0.10,
        "Rn_hi":   0.40,
        "Rt_mu":   0.74,     # mean tangential restitution
        "Rt_sig":  0.05,
        "Rt_lo":   0.50,
        "Rt_hi":   0.92,
        "color":   "#4393c3",  # blue
    },
    {
        "label":   "S  (100–500 kg)",
        "short":   "S",
        "mass_lo": 100,
        "mass_hi": 500,
        "m_rep":   250,
        "Rn_mu":  0.29,
        "Rn_sig": 0.05,
        "Rn_lo":  0.12,
        "Rn_hi":  0.42,
        "Rt_mu":  0.78,
        "Rt_sig": 0.05,
        "Rt_lo":  0.55,
        "Rt_hi":  0.93,
        "color":  "#74add1",  # light blue
    },
    {
        "label":   "M  (500–1 500 kg)",
        "short":   "M",
        "mass_lo": 500,
        "mass_hi": 1_500,
        "m_rep":   1_000,
        "Rn_mu":  0.32,
        "Rn_sig": 0.05,
        "Rn_lo":  0.14,
        "Rn_hi":  0.44,
        "Rt_mu":  0.81,
        "Rt_sig": 0.05,
        "Rt_lo":  0.58,
        "Rt_hi":  0.94,
        "color":  "#fee090",  # yellow
    },
    {
        "label":   "L  (1 500–3 000 kg)",
        "short":   "L",
        "mass_lo": 1_500,
        "mass_hi": 3_000,
        "m_rep":   2_000,
        "Rn_mu":  0.35,
        "Rn_sig": 0.05,
        "Rn_lo":  0.15,
        "Rn_hi":  0.46,
        "Rt_mu":  0.84,
        "Rt_sig": 0.05,
        "Rt_lo":  0.62,
        "Rt_hi":  0.95,
        "color":  "#f46d43",  # orange
    },
    {
        "label":   "XL  (3 000–10 000 kg)",
        "short":   "XL",
        "mass_lo": 3_000,
        "mass_hi": 10_000,
        "m_rep":   5_500,
        "Rn_mu":  0.38,
        "Rn_sig": 0.05,
        "Rn_lo":  0.18,
        "Rn_hi":  0.48,
        "Rt_mu":  0.87,
        "Rt_sig": 0.05,
        "Rt_lo":  0.65,
        "Rt_hi":  0.96,
        "color":  "#d73027",  # red
    },
]

print(f"{'Class':<24} {'m_rep (kg)':>10} {'Rn mean':>9} {'Rt mean':>9}")
print("-" * 55)
for sc in SIZE_CLASSES:
    print(f"{sc['label']:<24} {sc['m_rep']:>10,} {sc['Rn_mu']:>9.2f} {sc['Rt_mu']:>9.2f}")

In [ ]:
# Visualise the restitution parameter distributions for each class
fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.8))

x_eval = np.linspace(0.0, 1.0, 400)

for sc in SIZE_CLASSES:
    for ax, key_mu, key_sig, key_lo, key_hi, xlabel in [
        (axes[0], "Rn_mu", "Rn_sig", "Rn_lo", "Rn_hi", r"$R_n$"),
        (axes[1], "Rt_mu", "Rt_sig", "Rt_lo", "Rt_hi", r"$R_t$"),
    ]:
        from scipy.stats import truncnorm
        mu, sig = sc[key_mu], sc[key_sig]
        lo, hi  = sc[key_lo], sc[key_hi]
        a, b = (lo - mu) / sig, (hi - mu) / sig
        pdf = truncnorm.pdf(x_eval, a, b, loc=mu, scale=sig)
        ax.plot(x_eval, pdf, color=sc["color"], lw=2.0, label=sc["short"])
        ax.axvline(mu, color=sc["color"], lw=0.8, ls="--", alpha=0.6)

for ax, xlabel in zip(axes, [r"Normal restitution coefficient $R_n$",
                               r"Tangential restitution coefficient $R_t$"]):
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel("probability density")
    ax.legend(title="Size class", fontsize=10)

fig.suptitle("Restitution coefficient distributions by size class", fontweight="bold")
fig.tight_layout()
save_figure(fig, "06b_restitution_distributions")
plt.show()

## 4. Monte-Carlo simulation

For each size class we draw N = 500 pairs $(R_n, R_t)$ from the class-specific truncated normal distributions and run a trajectory for each pair. All trajectories start at the cliff top with a small horizontal push ($v_{x0}$ = 2 m/s), matching Notebook 06.

N = 500 per class (2 500 total) is sufficient to resolve the P95 runout to within ~20 m — adequate for the pedagogical comparisons we draw here.

In [ ]:
from scipy.stats import truncnorm

N_PER_CLASS = 500
VX0 = 2.0   # initial horizontal velocity [m/s] — same for all classes

results = {}   # key: class short label  →  dict with arrays

for sc in SIZE_CLASSES:
    label = sc["short"]

    def _sample(mu, sig, lo, hi):
        a, b = (lo - mu) / sig, (hi - mu) / sig
        return truncnorm.rvs(a, b, loc=mu, scale=sig,
                             size=N_PER_CLASS, random_state=RNG)

    Rn_arr = _sample(sc["Rn_mu"], sc["Rn_sig"], sc["Rn_lo"], sc["Rn_hi"])
    Rt_arr = _sample(sc["Rt_mu"], sc["Rt_sig"], sc["Rt_lo"], sc["Rt_hi"])

    stop_x  = np.empty(N_PER_CLASS)
    n_bnc   = np.empty(N_PER_CLASS, dtype=int)
    paths   = []   # store a subset for trajectory overlay

    for i in range(N_PER_CLASS):
        xs, ys, bnc = simulate_rockfall(
            x0=source_x, y0=source_y,
            vx0=VX0, vy0=0.0,
            terrain_x=terrain_x, terrain_y=terrain_y,
            R_n=Rn_arr[i], R_t=Rt_arr[i],
        )
        stop_x[i] = xs[-1]
        n_bnc[i]  = len(bnc)
        if i < 40:
            paths.append((xs, ys))

    results[label] = {
        "sc":     sc,
        "Rn":     Rn_arr,
        "Rt":     Rt_arr,
        "stop_x": stop_x,
        "n_bnc":  n_bnc,
        "paths":  paths,
    }

    p50 = np.median(stop_x)
    p95 = np.percentile(stop_x, 95)
    print(f"{sc['label']:<24}  median = {p50:5.0f} m   P95 = {p95:5.0f} m   "
          f"mean bounces = {n_bnc.mean():.1f}")

## 5. Trajectory overlays — one panel per size class

The first 40 trajectories from each class are overlaid on the terrain profile. Note how the trajectory cloud shifts downslope and the individual paths become more variable as block size increases — larger blocks are more sensitive to the exact combination of $(R_n, R_t)$ drawn.

In [ ]:
fig, axes = plt.subplots(len(SIZE_CLASSES), 1,
                          figsize=(11.0, 4.0 * len(SIZE_CLASSES)),
                          sharex=True, sharey=True)

for ax, sc_label in zip(axes, [sc["short"] for sc in SIZE_CLASSES]):
    res = results[sc_label]
    sc  = res["sc"]

    # Terrain
    ax.fill_between(terrain_x, terrain_y, -30, color=COLORS["soil"], alpha=0.22)
    ax.plot(terrain_x, terrain_y, color=COLORS["soil"], lw=2.0)

    # Trajectory subset
    for xs, ys in res["paths"]:
        ax.plot(xs, ys, color=sc["color"], lw=0.7, alpha=0.25)

    # Stopping points
    sx = res["stop_x"]
    ax.plot(sx, np.interp(sx, terrain_x, terrain_y),
            "|", ms=8, color=sc["color"], alpha=0.35, markeredgewidth=1.2)

    # Median and P95 lines
    p50 = np.median(sx)
    p95 = np.percentile(sx, 95)
    ax.axvline(p50, color=sc["color"], lw=1.8, ls="--",
               label=f"median {p50:.0f} m")
    ax.axvline(p95, color=sc["color"], lw=1.8, ls=":",
               label=f"P95 {p95:.0f} m")

    ax.plot([source_x], [source_y], "o", ms=9, color="black")
    ax.set_xlim(0, 1300)
    ax.set_ylim(-30, 680)
    ax.set_ylabel("elevation [m]")
    ax.set_title(sc["label"], loc="left", fontweight="bold")
    ax.legend(loc="upper right", fontsize=10)

axes[-1].set_xlabel("horizontal distance from cliff [m]")
fig.suptitle("Monte-Carlo trajectory ensembles by size class  "
             "(first 40 paths shown, all stopping points marked)",
             y=1.002, fontweight="bold")
fig.tight_layout()
save_figure(fig, "06b_trajectory_panels")
plt.show()

## 6. Runout comparison — box plots and histograms

Box plots compress the full stopping-point distribution for each class into five numbers: minimum, Q1, median, Q3, and maximum (with outliers). They make it easy to see both the **shift** in median runout and the **spread** within each class.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.5))

labels     = [sc["short"] for sc in SIZE_CLASSES]
stop_arrays = [results[l]["stop_x"] for l in labels]
colors_list = [sc["color"] for sc in SIZE_CLASSES]

# ---- Box plot ----
ax = axes[0]
bp = ax.boxplot(stop_arrays, labels=labels, patch_artist=True,
                medianprops=dict(color="white", lw=2.5),
                flierprops=dict(marker=".", markersize=3, alpha=0.4))
for patch, col in zip(bp["boxes"], colors_list):
    patch.set_facecolor(col)
    patch.set_alpha(0.75)
for whisker, cap in zip(bp["whiskers"], bp["caps"]):
    whisker.set_color("dimgray")
    cap.set_color("dimgray")
ax.set_xlabel("Size class"); ax.set_ylabel("Runout [m]")
ax.set_title("Runout distribution by size class")

# ---- Overlapping histograms ----
ax = axes[1]
bins = np.linspace(0, 1350, 40)
for l, col in zip(labels, colors_list):
    ax.hist(results[l]["stop_x"], bins=bins,
            color=col, alpha=0.45, label=l, density=True)
    # Median tick
    med = np.median(results[l]["stop_x"])
    ax.axvline(med, color=col, lw=1.8, ls="--")
ax.set_xlabel("Runout [m]"); ax.set_ylabel("Density")
ax.set_title("Runout histograms (dashed = median)")
ax.legend(title="Size class", fontsize=10)

fig.tight_layout()
save_figure(fig, "06b_runout_comparison")
plt.show()

# Print summary table
print(f"{'Class':<24} {'P10':>7} {'P50':>7} {'P90':>7} {'P95':>7} {'P99':>7}  [m]")
print("-" * 60)
for l in labels:
    sx = results[l]["stop_x"]
    vals = np.percentile(sx, [10, 50, 90, 95, 99])
    print(f"{results[l]['sc']['label']:<24}  "
          + "  ".join(f"{v:5.0f}" for v in vals))

## 7. Runout-exceedance curves by size class

Each curve shows the fraction of trajectories in that class that reached at least a given horizontal distance. Reading across horizontally at any exceedance probability directly gives the design runout for that probability level — the conventional input to a hazard-zone boundary.

Plotting all five classes on one panel reveals how the entire distribution shifts and how the tails diverge. The P99 runout of the XL class defines the worst-credible scenario when large blocks are possible from the source.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.5))

for l in labels:
    sx  = np.sort(results[l]["stop_x"])
    exc = 1.0 - np.arange(len(sx)) / len(sx)
    col = results[l]["sc"]["color"]
    ax.step(sx, exc, where="post", color=col, lw=2.0,
            label=results[l]["sc"]["label"])

# Hazard-zone bands (conventional thresholds)
ax.axhspan(1e-1, 1.0,  alpha=0.08, color=COLORS["fail"],   zorder=0)
ax.axhspan(1e-2, 1e-1, alpha=0.08, color=COLORS["accent"], zorder=0)
ax.text(30, 0.35, "High hazard\n(> 10⁻¹)",  fontsize=10, color=COLORS["fail"],   alpha=0.9)
ax.text(30, 0.04, "Moderate\n(10⁻² – 10⁻¹)", fontsize=10, color=COLORS["accent"], alpha=0.9)

ax.set_yscale("log")
ax.set_ylim(5e-3, 1.1)
ax.set_xlim(0, 1350)
ax.set_xlabel("horizontal distance from cliff [m]", fontsize=12)
ax.set_ylabel(r"P(runout $\geq$ x)", fontsize=12)
ax.set_title("Runout exceedance curves by size class", fontweight="bold")
ax.legend(title="Size class", fontsize=10, loc="upper right")
save_figure(fig, "06b_exceedance_curves")
plt.show()

## 8. Number of bounces and kinetic energy at stop

Two diagnostics that matter for engineering design:

- **Number of bounces** — relevant for structural design of rockfall fences (multiple-impact loading vs. single-impact); larger blocks tend to bounce fewer times because each impact is relatively more energetic and the block retains enough speed to keep rolling.
- **Bounce count trends** — the variation across classes is a reminder that the *dynamics* differ, not just the final stopping point.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8))

# ---- Bounce count box plot ----
ax = axes[0]
bnc_arrays = [results[l]["n_bnc"] for l in labels]
bp = ax.boxplot(bnc_arrays, labels=labels, patch_artist=True,
                medianprops=dict(color="white", lw=2.5),
                flierprops=dict(marker=".", markersize=3, alpha=0.4))
for patch, col in zip(bp["boxes"], colors_list):
    patch.set_facecolor(col); patch.set_alpha(0.75)
for whisker, cap in zip(bp["whiskers"], bp["caps"]):
    whisker.set_color("dimgray"); cap.set_color("dimgray")
ax.set_xlabel("Size class"); ax.set_ylabel("Number of bounces")
ax.set_title("Bounce count per trajectory")

# ---- Runout vs. Rt scatter, coloured by class ----
ax = axes[1]
for l in labels:
    res = results[l]
    col = res["sc"]["color"]
    ax.scatter(res["Rt"], res["stop_x"],
               color=col, alpha=0.18, s=18,
               label=res["sc"]["short"])
    # Regression line
    m, b = np.polyfit(res["Rt"], res["stop_x"], 1)
    x_fit = np.array([res["Rt"].min(), res["Rt"].max()])
    ax.plot(x_fit, m * x_fit + b, color=col, lw=2.0)

ax.set_xlabel(r"Tangential restitution $R_t$")
ax.set_ylabel("Runout [m]")
ax.set_title(r"Runout vs $R_t$ — each class separately")
ax.legend(title="Class", fontsize=9)

fig.tight_layout()
save_figure(fig, "06b_bounce_count_and_scatter")
plt.show()

print(f"\n{'Class':<24} {'Mean bounces':>13} {'Median bounces':>15}")
print("-" * 54)
for l in labels:
    nb = results[l]["n_bnc"]
    print(f"{results[l]['sc']['label']:<24}  {nb.mean():>12.1f}  {np.median(nb):>14.0f}")

## 9. Mixed-source scenario: what if all block sizes are equally probable?

In practice a cliff releases blocks of all sizes. If we assume the five classes are equally probable (20 % each — a simplification; real source-area inventories show a power-law mass-frequency distribution), we can pool all 2 500 trajectories to get a **mixed-source exceedance curve** and compare it with the class-specific curves.

The mixed-source P95 is pulled toward the XL class — the rare large block dominates the tail. This is why hazard maps based only on the most common (small) block size systematically underestimate the design runout.

In [ ]:
all_stop_x = np.concatenate([results[l]["stop_x"] for l in labels])

fig, ax = plt.subplots(figsize=(9.5, 5.5))

# Individual class curves (light)
for l in labels:
    sx  = np.sort(results[l]["stop_x"])
    exc = 1.0 - np.arange(len(sx)) / len(sx)
    col = results[l]["sc"]["color"]
    ax.step(sx, exc, where="post", color=col, lw=1.2, alpha=0.55,
            label=results[l]["sc"]["label"])

# Mixed-source curve (bold black)
sx_all  = np.sort(all_stop_x)
exc_all = 1.0 - np.arange(len(sx_all)) / len(sx_all)
ax.step(sx_all, exc_all, where="post", color="black", lw=2.8,
        label=f"Mixed source (equal weighting, N = {len(all_stop_x):,})")

# Percentile markers
for p, ls in [(50, "--"), (95, ":"), (99, "-.")]:
    xp = np.percentile(all_stop_x, p)
    ax.axvline(xp, color="black", lw=1.2, ls=ls)
    ax.text(xp + 8, 0.85, f"P{p} = {xp:.0f} m",
            fontsize=10, va="top", rotation=90)

ax.set_yscale("log")
ax.set_ylim(5e-3, 1.1)
ax.set_xlim(0, 1350)
ax.set_xlabel("horizontal distance from cliff [m]", fontsize=12)
ax.set_ylabel(r"P(runout $\geq$ x)", fontsize=12)
ax.set_title("Mixed-source exceedance curve vs. individual class curves", fontweight="bold")
ax.legend(fontsize=9, loc="lower left")
save_figure(fig, "06b_mixed_source_exceedance")
plt.show()

print("Mixed-source runout percentiles:")
for p in [50, 90, 95, 99]:
    print(f"  P{p:2d} = {np.percentile(all_stop_x, p):.0f} m")

## 10. Discussion

### What drives the size–runout relationship in this model?

Mass itself is absent from the equations of free flight — the acceleration is always $g$ regardless of block size. The longer runout of large blocks emerges entirely from the **mass-dependent restitution coefficients**:

- The $R_t$ increase from 0.74 (XS) to 0.87 (XL) means that each bounce bleeds off less tangential momentum for a large block. Over 20–40 bounces the compounding effect is large.
- The smaller $R_n$ increase (0.27 → 0.38) also lengthens the bounce arc slightly, putting the block on a flatter post-impact trajectory that carries it further before the next ground contact.

This is consistent with back-analysis of real events: the dominant control on runout is how efficiently tangential momentum is preserved at each impact, not the free-flight kinematics.

### Implications for hazard mapping

1. **Source characterisation matters.** If you know the cliff can only release small blocks (e.g., a thin, vertically jointed limestone band), using XL restitution parameters would overestimate the hazard. Conversely, if large block releases are credible (granite exfoliation, large joint-bounded wedges), using small-block parameters is unconservative.
2. **The rare large block sets the outer hazard boundary.** Even if 90 % of rockfalls are class XS, the P99 runout — which controls where no buildings should be placed — is dominated by the 10 % that are class XL or larger.
3. **Equal-weighting is usually wrong.** Real mass-frequency distributions in rock slopes follow an approximate power law (many small blocks, few large ones). A realistic mixed-source model weights the XL class at < 1 % but gives it a disproportionate influence on the exceedance tail.

### Limitations of this experiment

- **Mass cancels in free flight.** A more physically complete model would include rotational inertia (which *does* depend on block shape and mass distribution) and would couple rotation and translation explicitly.
- **The $R_n$/$R_t$–mass relationships used here are approximate.** Operational codes derive these empirically from site-specific field experiments; transferring regional averages to a specific cliff introduces systematic uncertainty.
- **Fragmentation is neglected.** Large blocks often shatter on first impact, generating a shower of fragments that travel further than the intact parent would have. This would further increase the effective runout of the XL class.
- **Terrain is 2-D.** Real topography channels and deflects rockfalls laterally in ways a slope-profile model cannot capture.

## Take-aways

- In a lumped-mass model, **mass does not appear in the equations of motion** — the size effect is encoded entirely in the restitution coefficients.
- Larger blocks have **higher $R_t$** (they override surface roughness) and **slightly higher $R_n$** (stiffer elastic response), producing systematically longer runout.
- The P95 runout increases by roughly 20–30 % from XS to XL class across this terrain — a difference that straddles typical hazard-zone boundaries.
- For a mixed-size source, the **rare large block dominates the tail** of the runout distribution and therefore controls the outer red-zone boundary.
- Source characterisation — knowing what block sizes a cliff can realistically produce — is as important as the restitution parameters in a probabilistic rockfall hazard assessment.

## Voluntary exercises

1. The restitution parameters in Section 3 are for an alpine talus surface. How would you expect them to change (and the class gaps to narrow or widen) if the surface were a dense spruce forest? Modify the class definitions and re-run Section 4–7 to test your hypothesis.
2. Replace the equal-weighting assumption in Section 9 with a power-law mass-frequency distribution $P(m) \propto m^{-\alpha}$ with $\alpha$ = 1.5 (a common empirical value for rock-slope inventories). How does the mixed-source P95 change?
3. The XS class has more bounces but shorter runout than the XL class. Explain this qualitatively in terms of the energy budget at each impact.
4. In a real hazard assessment you would sample block mass as a continuous variable, not in five discrete classes. Sketch (in words or pseudocode) how you would implement a continuous mass-dependent $R_n(m)$ and $R_t(m)$ parameterisation using the five class means as anchor points.

## References

- Bourrier, F., Dorren, L., Nicot, F., Berger, F., & Darve, F. (2009). *Toward objective rockfall trajectory simulation using a stochastic impact model.* Geomorphology, 110(3–4), 68–79.
- Dorren, L. K. A. (2003). *A review of rockfall mechanics and modelling approaches.* Progress in Physical Geography, 27(1), 69–87.
- Dorren, L. K. A., Maier, B., Putters, U. S., & Seijmonsbergen, A. C. (2004). *Combining field and modelling techniques to assess rockfall dynamics on a protection forest hillslope in the European Alps.* Geomorphology, 57(3–4), 151–167.
- Pfeiffer, T. J. & Bowen, T. D. (1989). *Computer simulation of rockfalls.* Bulletin of the Association of Engineering Geologists, 26(1), 135–146.
- Agliardi, F. & Crosta, G. B. (2003). *High resolution three-dimensional numerical modelling of rockfalls.* International Journal of Rock Mechanics and Mining Sciences, 40(4), 455–471.